# Solar Active-Region Detection — Kaggle training run (MAX-QUALITY preset)

**One cell to run** (safe to re-run any time: it stops the old run, re-clones
the small code repo, and resumes data + training from where they stopped).

Right-hand panel (**Session options**):

1. **Internet: ON** — required for the download.
2. **GPU: T4 x2** — both are used automatically (DataParallel).
3. **Persistence: Files only** — WITHOUT this, each 12-hour session end WIPES
   your data + model. Set it while a session is running for it to apply.

Preset (T4x2 runtime): `BASE_CHANNELS=48 DEEP_SUPERVISION=1` (best-quality 18M
model, batch auto-fits VRAM), 16 download workers, 2,000 frames, every core.

When a session ends (12 h): run this same cell again in the same notebook — it
resumes. The newest model is mirrored to /kaggle/output (Output tab) every 5 min.

In [ ]:
%%bashset -xexport HOME=/kaggle/workingcd /kaggle/working# 1) Stop any previous run of this notebook (lock PID + trainer + downloader),#    so a re-run starts clean:P=$(cat /kaggle/working/solar_results/arpil/run_forever.lock 2>/dev/null)[ -n "$P" ] && kill -TERM "$P" 2>/dev/nullpkill -f "SOALR/scripts/train_streaming.py" 2>/dev/nullpkill -f "SOALR/scripts/build_arpil_resumable.py" 2>/dev/nullsleep 5pkill -9 -f "SOALR/scripts/train_streaming.py" 2>/dev/nullpkill -9 -f "SOALR/scripts/build_arpil_resumable.py" 2>/dev/null# 2) Fresh code repo (tiny, ~15 s). Your DATA + CHECKPOINTS live OUTSIDE it#    (/kaggle/working/solar_data and solar_results) and are never touched.#    Re-cloning sidesteps any dirty git state left by an earlier attempt:rm -rf /kaggle/work SOALRgit clone -q -b arena/01a04247-soalr-active-region-detection \    https://github.com/haydenCoder/SOALR-ACTIVE-REGION-DETECTION-MODEL-SUN-.git SOALR \|| { mkdir -p SOALR && wget -qO /tmp/repo.tgz \    https://codeload.github.com/haydenCoder/SOALR-ACTIVE-REGION-DETECTION-MODEL-SUN-/tar.gz/refs/heads/arena/01a04247-soalr-active-region-detection \    && tar xzf /tmp/repo.tgz -C SOALR --strip-components=1; }if [ ! -x SOALR/scripts/run_forever.sh ]; then    echo "STOP: repository download failed. In the notebook Settings, make sure Internet is ON, then re-run this cell."    exit 1ficd SOALR# 3) NO venv on Kaggle (ensurepip is broken there). Install the small missing#    packages straight into the session environment; torch is preinstalled.python3 -m pip install -q -r requirements.txtpython3 -c "import torch; print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())"# 4) Mirror the newest model + log to /kaggle/output every 5 min (that is the#    only folder you can DOWNLOAD from, via the Output tab after a session ends):mkdir -p /kaggle/output(    while :; do        sleep 300        cp -f /kaggle/working/solar_results/arpil/continuous/best.pt    /kaggle/output/ 2>/dev/null        cp -f /kaggle/working/solar_results/arpil/continuous/last.pt    /kaggle/output/ 2>/dev/null        cp -f /kaggle/working/solar_results/arpil/continuous/metrics.jsonl /kaggle/output/ 2>/dev/null        cp -f /kaggle/working/solar_results/arpil/run_forever.log      /kaggle/output/ 2>/dev/null        cp -f /kaggle/working/solar_results/arpil/STATUS.md            /kaggle/output/ 2>/dev/null    done) &SAVE_WATCHER=$!# 5) MAX-QUALITY preset for Kaggle (both T4s used automatically via DataParallel):CHANNELS="aia94 aia131 aia1600 aia171 aia193 aia211 aia304 aia335 hmi_m hmi_bx hmi_by hmi_bz hmi_v" \SOLAR_PYTHON="$(command -v python3)" BASE_CHANNELS=48 DEEP_SUPERVISION=1 \MIN_FREE_GB=4 MAX_TOTAL_FRAMES=2000 FRAMES_PER_CYCLE=200 DOWNLOAD_WORKERS=16 \CPU_HEADROOM=0 TILES_PER_EPOCH=200 VAL_EPOCH=10 VAL_SUBSET=300 \bash scripts/run_forever.shkill "$SAVE_WATCHER" 2>/dev/null

## What you will see

- `torch 2.x.x | CUDA available: True | GPUs: 2` — environment healthy
- Preflight `3 ok`, then `Downloading 200 frames with 16 parallel workers ...`
- `[stream] using 2 GPUs with DataParallel (each batch splits across both)`
- `[stream] epoch=0001 loss=0.7x ...` — finite loss; first `val_dice` at epoch 10

## Re-running

This cell is idempotent: it stops any running instance, re-clones the code
(~15 s), and resumes. With **Persistence: Files only**, all downloaded data and
checkpoints survive — only the code is refreshed.